In [1]:
print("Starting to Download")

Starting to Download


In [2]:
import os
os.environ['AWS_NO_SIGN_REQUEST'] = 'YES'

In [3]:
!uv pip install pystac_client s3fs stackstac shapely geopandas rioxarray rasterio pyproj scipy

Audited 9 packages in 67ms


In [4]:
import stackstac
from pystac_client import Client
from pystac import Item
import datetime
import json
import numpy as np
import pandas as pd
import geopandas as gpd
import rioxarray
import rasterio
from rasterio.merge import merge as rio_merge
from rasterio.mask import mask
from rasterio.warp import reproject, Resampling, calculate_default_transform
from rasterio.enums import Resampling
from shapely.geometry import shape, box, mapping
from shapely.ops import transform
import pyproj
import s3fs
import os
from pathlib import Path
import time
from concurrent.futures import ThreadPoolExecutor, as_completed
from scipy.ndimage import zoom
import warnings
warnings.filterwarnings('ignore')

print("All libraries imported successfully!")

All libraries imported successfully!


In [5]:
# Connect to Earth Search API
API_URL = "https://earth-search.aws.element84.com/v1"
catalog = Client.open(API_URL)
print("Connected to STAC API")

Connected to STAC API


In [6]:
# Load your GeoJSON
with open('Final_AOI_Leh.geojson') as f:
    aoi_geojson = json.load(f)

# Extract geometry
if aoi_geojson.get('type') == 'FeatureCollection':
    feature = aoi_geojson['features'][0]
    geometry = feature['geometry']
else:
    geometry = aoi_geojson['geometry']

# Convert to shapely geometry
aoi_geom = shape(geometry)
print(f"AOI Type: {aoi_geom.geom_type}")
print(f"AOI Bounds: {aoi_geom.bounds}")

# Determine the correct UTM zone for AOI
centroid_lon = aoi_geom.centroid.x
utm_zone = int((centroid_lon + 180) / 6) + 1
target_epsg = 32600 + utm_zone  # Northern hemisphere
print(f"Target CRS: EPSG:{target_epsg} (UTM Zone {utm_zone}N)")

AOI Type: MultiPolygon
AOI Bounds: (8670863.441561064, 3946524.062684501, 9022722.685177073, 4357758.018325165)
Target CRS: EPSG:1507409 (UTM Zone 1474809N)


In [7]:
# Define date range for search
start_date = datetime.datetime(2025, 5, 1)
end_date = datetime.datetime(2025, 8, 31)
time_range = f"{start_date.isoformat()}Z/{end_date.isoformat()}Z"

print(f"Searching from {start_date.date()} to {end_date.date()}")

Searching from 2025-05-01 to 2025-08-31


In [8]:
from shapely.ops import transform
import pyproj

# Reproject AOI from 3857 to 4326 (WGS84) once
project_to_wgs84 = pyproj.Transformer.from_crs(
    'epsg:3857', 'epsg:4326', always_xy=True
).transform
aoi_wgs84 = transform(project_to_wgs84, aoi_geom)

# Then use the simplified function
def search_and_filter_items_simple(catalog, aoi_geom, start_date, end_date, max_cloud=20):
    """
    Search for Sentinel-2 items and filter to only those overlapping the AOI
    Assumes both AOI and item geometries are in WGS84 (EPSG:4326)
    """
    # Get AOI bounds
    bbox_wgs84 = list(aoi_geom.bounds)

    # Search
    search = catalog.search(
        collections=["sentinel-2-l2a"],
        bbox=bbox_wgs84,
        datetime=f"{start_date.isoformat()}Z/{end_date.isoformat()}Z",
        query={"eo:cloud_cover": {"lt": max_cloud}},
        max_items=500
    )

    all_items = list(search.get_all_items())
    print(f"Found {len(all_items)} scenes before filtering")

    # Filter by spatial overlap
    filtered_items = []
    skipped_items = []

    for item in all_items:
        try:
            # Get item geometry (in WGS84)
            item_geom = shape(item.geometry)

            # Check if it overlaps with AOI (both in WGS84 now)
            if item_geom.intersects(aoi_geom):
                filtered_items.append(item)
            else:
                # Check bbox overlap (faster check)
                item_bbox = box(*item.bbox)
                if item_bbox.intersects(aoi_geom):
                    filtered_items.append(item)
                else:
                    skipped_items.append(item.id)
        except Exception as e:
            print(f"⚠️ Error checking item {item.id}: {e}")
            skipped_items.append(item.id)

    print(f"Found {len(filtered_items)} scenes overlapping AOI")
    if skipped_items:
        print(f"⚠️ Skipped {len(skipped_items)} tiles that don't overlap AOI")

    return filtered_items

# Use with reprojected AOI
filtered_items = search_and_filter_items_simple(catalog, aoi_wgs84, start_date, end_date)

Found 293 scenes before filtering
Found 273 scenes overlapping AOI
⚠️ Skipped 20 tiles that don't overlap AOI


In [9]:
def group_items_by_date(items):
    """Group STAC items by acquisition date (no overlap check needed)"""
    date_groups = {}

    for item in items:
        try:
            # Extract date from item ID (format: S2*_44SKE_20250620_0_L2A)
            try:
                date_str = item.id.split('_')[2]
                date = datetime.datetime.strptime(date_str, '%Y%m%d').date()
            except:
                # Fallback to datetime property
                date_str = item.properties.get('datetime', '')[:10]
                date = datetime.datetime.strptime(date_str, '%Y-%m-%d').date()

            if date not in date_groups:
                date_groups[date] = []
            date_groups[date].append(item)

        except Exception as e:
            print(f"⚠️ Error processing {item.id}: {e}")

    return date_groups

# Simply group by date (no need to re-filter)
date_groups = group_items_by_date(filtered_items)
sorted_dates = sorted(date_groups.keys())

print(f"\n📊 Found {len(sorted_dates)} unique dates with overlapping tiles")
for date in sorted_dates[:5]:  # Show first 5
    tile_count = len(date_groups[date])
    tiles = [item.id.split('_')[1] for item in date_groups[date]]
    print(f"  {date}: {tile_count} tiles - {', '.join(tiles)}")
if len(sorted_dates) > 5:
    print(f"  ... and {len(sorted_dates) - 5} more dates")


📊 Found 39 unique dates with overlapping tiles
  2025-05-01: 3 tiles - 43SGV, 44SKE, 43SGA
  2025-05-05: 3 tiles - 44SMD, 44SND, 44SLE
  2025-05-06: 4 tiles - 43SGV, 44SKE, 43SGA, 44SKF
  2025-05-08: 10 tiles - 43SGU, 44SKD, 43SGV, 44SKE, 44SLE, 44SME, 43SGA, 44SKF, 44SLF, 44SMF
  2025-05-13: 1 tiles - 43SGS
  ... and 34 more dates


In [10]:
def resample_array_simple(array, target_shape):
    """Simple array resampling using scipy"""
    height, width = array.shape
    target_height, target_width = target_shape[1], target_shape[2]

    # Only resample if different
    if height == target_height and width == target_width:
        return array

    zoom_y = target_height / height
    zoom_x = target_width / width

    return zoom(array, (zoom_y, zoom_x), order=1)

def merge_tiles_for_date_safe(items, aoi_geom, target_epsg):
    """
    Merge tiles with safe reprojection.
    Only processes tiles that can be reprojected to the target CRS.
    """
    if len(items) == 0:
        return None, None

    print(f"  Target CRS: EPSG:{target_epsg}")

    # Define bands
    band_assets = ['blue-jp2', 'green-jp2', 'red-jp2', 'nir-jp2',
                   'nir08-jp2', 'swir16-jp2', 'swir22-jp2', 'scl-jp2']
    band_names = ['blue', 'green', 'red', 'nir', 'nir08', 'swir16', 'swir22', 'scl']

    # Convert AOI to GeoDataFrame with proper CRS
    aoi_gdf = gpd.GeoDataFrame(geometry=[aoi_geom], crs='epsg:4326')

    tile_data_list = []
    processed_tiles = []

    for item in items:
        try:
            tile_id = item.id.split('_')[1]
            print(f"  Processing tile: {tile_id}")

            # Try to stack with target CRS
            try:
                data = stackstac.stack(
                    [item],
                    assets=band_assets,
                    resolution=10,
                    epsg=target_epsg,
                    bounds_latlon=aoi_geom.bounds,
                    chunksize=(1, 1024, 1024)
                )
            except Exception as e:
                # If target CRS fails, try native CRS
                print(f"    ⚠️ Target CRS failed, trying native CRS...")
                data = stackstac.stack(
                    [item],
                    assets=band_assets,
                    resolution=10,
                    bounds_latlon=aoi_geom.bounds,
                    chunksize=(1, 1024, 1024)
                )

            # Clip to AOI using the GeoDataFrame
            data_clipped = data.rio.clip(aoi_gdf.geometry, crs="EPSG:4326", all_touched=True)
            data_computed = data_clipped.compute()

            # Extract bands
            if len(data_computed.band) > 1:
                bands = data_computed.values[0]  # (band, y, x)
                tile_data_list.append(bands)
                processed_tiles.append(tile_id)
            else:
                bands = data_computed.values[0]
                tile_data_list.append(bands)
                processed_tiles.append(tile_id)

        except Exception as e:
            print(f"    ❌ Skipping {item.id}: {str(e)[:100]}")
            continue

    if len(tile_data_list) == 0:
        print("  ❌ No tiles could be processed")
        return None, None

    print(f"  ✅ Successfully processed {len(tile_data_list)} tiles")

    # Merge tiles
    if len(tile_data_list) == 1:
        merged_bands = tile_data_list[0]
    else:
        # Find target shape (use the largest)
        shapes = [arr.shape for arr in tile_data_list]
        target_shape = max(shapes, key=lambda x: x[1] * x[2])

        # Resample all tiles to target shape
        resampled_tiles = []
        for bands in tile_data_list:
            if bands.shape != target_shape:
                resampled = np.zeros(target_shape, dtype='float32')
                for b in range(bands.shape[0]):
                    resampled[b] = resample_array_simple(bands[b], target_shape)
                resampled_tiles.append(resampled)
            else:
                resampled_tiles.append(bands)

        # Merge by taking mean (or max) of overlapping areas
        merged_bands = np.mean(resampled_tiles, axis=0)

    return merged_bands, band_names

In [11]:
def analyze_single_tile_dates(date_groups):
    """Analyze single-tile dates and print summary"""

    if not date_groups:
        print("⚠️ No date groups to process")
        return

    single_dates = []
    multi_dates = []

    for date, items in date_groups.items():
        if len(items) == 1:
            single_dates.append(date)
        else:
            multi_dates.append((date, len(items)))

    print(f"\n📊 Date Analysis:")
    print(f"  Total dates: {len(date_groups)}")
    print(f"  Dates with single tile: {len(single_dates)}")
    print(f"  Dates with multiple tiles: {len(multi_dates)}")

    if single_dates:
        print(f"\n📅 Single-tile dates:")
        for date in sorted(single_dates):
            item = date_groups[date][0]
            tile_name = item.id.split('_')[1] if '_' in item.id else 'unknown'
            cloud = item.properties.get('eo:cloud_cover', 'N/A')
            print(f"  {date}: {tile_name} (cloud cover: {cloud}%)")

    if multi_dates:
        print(f"\n📅 Multi-tile dates:")
        for date, count in sorted(multi_dates):
            print(f"  {date}: {count} tiles")

    return single_dates, multi_dates

# Analyze the date groups
single_dates, multi_dates = analyze_single_tile_dates(date_groups)


📊 Date Analysis:
  Total dates: 39
  Dates with single tile: 9
  Dates with multiple tiles: 30

📅 Single-tile dates:
  2025-05-13: 43SGS (cloud cover: 17.851138%)
  2025-05-20: 44SLB (cloud cover: 15.451418%)
  2025-05-26: 43SGV (cloud cover: 16.954215%)
  2025-06-29: 44SLD (cloud cover: 17.494304%)
  2025-07-14: 44SNF (cloud cover: 18.581067%)
  2025-08-03: 44SNB (cloud cover: 17.132802%)
  2025-08-09: 43SGV (cloud cover: 15.488492%)
  2025-08-15: 44SNB (cloud cover: 10.836514%)
  2025-08-26: 43SGA (cloud cover: 17.363802%)

📅 Multi-tile dates:
  2025-05-01: 3 tiles
  2025-05-05: 3 tiles
  2025-05-06: 4 tiles
  2025-05-08: 10 tiles
  2025-05-18: 9 tiles
  2025-05-23: 21 tiles
  2025-05-27: 8 tiles
  2025-06-07: 19 tiles
  2025-06-09: 14 tiles
  2025-06-10: 2 tiles
  2025-06-12: 17 tiles
  2025-06-15: 2 tiles
  2025-06-19: 26 tiles
  2025-06-20: 4 tiles
  2025-06-22: 10 tiles
  2025-06-25: 4 tiles
  2025-06-27: 4 tiles
  2025-07-02: 8 tiles
  2025-07-04: 14 tiles
  2025-07-06: 14 tile

In [31]:
import stackstac
import geopandas as gpd
import numpy as np
from shapely.geometry import shape, box, MultiPolygon
import datetime
import rioxarray
import xarray as xr
from shapely.ops import transform
import pyproj
import warnings
warnings.filterwarnings('ignore')

from concurrent.futures import ThreadPoolExecutor, as_completed
from functools import lru_cache
import os

# --- Set AWS_NO_SIGN_REQUEST for public S3 buckets ---
os.environ['AWS_NO_SIGN_REQUEST'] = 'YES'

# --- In-memory cache for band data ---
# Key: (item_id, band_name) -> numpy array (2D)
_band_cache = {}
CACHE_MAX_SIZE = 256  # limit to avoid memory blow-up

def get_band_data(item, band_name):
    """Load band data with caching."""
    key = (item.id, band_name)
    if key in _band_cache:
        return _band_cache[key]
    
    # Load the band
    try:
        url = item.assets[band_name].href
        ds = xr.open_dataset(url, engine='rasterio')
        
        # Extract data (2D)
        if hasattr(ds, 'band_data'):
            data = ds.band_data.values
        elif hasattr(ds, 'data'):
            data = ds.data.values
        else:
            data_vars = list(ds.data_vars.keys())
            if data_vars:
                data = ds[data_vars[0]].values
            else:
                data = ds.values
        
        if len(data.shape) == 3:
            # Take first band if it's a single band, otherwise take all? 
            # For safety, if shape[0] == 1, take that; else we need to decide.
            # Here we take first band (assuming it's the band we want)
            data = data[0] if data.shape[0] == 1 else data
        ds.close()
        
        # Store in cache (with size limit)
        if len(_band_cache) >= CACHE_MAX_SIZE:
            # Remove a random item (simple FIFO)
            _band_cache.pop(next(iter(_band_cache)))
        _band_cache[key] = data
        return data
    except Exception as e:
        # Fallback to rasterio
        try:
            import rasterio
            url = item.assets[band_name].href
            with rasterio.open(url) as src:
                data = src.read(1)
            if data is not None and data.size > 0:
                if len(_band_cache) >= CACHE_MAX_SIZE:
                    _band_cache.pop(next(iter(_band_cache)))
                _band_cache[key] = data
                return data
        except Exception as e2:
            print(f"    ⚠️ Failed to load {band_name} for {item.id}: {e2}")
            return None

def process_single_date(date, items, aoi_geom, target_epsg):
    """Process one date: load all bands for its single tile in parallel."""
    if len(items) != 1:
        return None
    
    item = items[0]
    tile_crs = item.properties.get('proj:epsg', target_epsg)
    
    # Find all -jp2 assets, excluding visual
    available_bands = [key for key in item.assets.keys() 
                       if key.endswith('-jp2') and 'visual' not in key]
    if not available_bands:
        print(f"  ⚠️ No band assets for {date}")
        return None
    
    # Load bands in parallel using threads
    band_data = {}
    with ThreadPoolExecutor(max_workers=len(available_bands)) as executor:
        future_to_band = {
            executor.submit(get_band_data, item, band): band 
            for band in available_bands
        }
        for future in as_completed(future_to_band):
            band = future_to_band[future]
            try:
                data = future.result()
                if data is not None:
                    band_data[band] = data
            except Exception as e:
                print(f"    ⚠️ Error in thread for {band}: {e}")
    
    if not band_data:
        print(f"  ⚠️ No bands loaded for {date}")
        return None
    
    # Reorder bands according to available_bands (keep consistent order)
    ordered_data = [band_data[b] for b in available_bands if b in band_data]
    if not ordered_data:
        return None
    
    # Stack bands
    if len(ordered_data) == 1:
        data_array = ordered_data[0]
    else:
        # Ensure all shapes match; if not, crop to smallest (simple solution)
        shapes = [d.shape for d in ordered_data]
        if len(set(shapes)) == 1:
            data_array = np.stack(ordered_data, axis=0)
        else:
            min_shape = tuple(min(dim) for dim in zip(*shapes))
            resized = []
            for d in ordered_data:
                if d.shape != min_shape:
                    slices = tuple(slice(0, s) for s in min_shape)
                    resized.append(d[slices])
                else:
                    resized.append(d)
            data_array = np.stack(resized, axis=0)
    
    return {
        'date': date,
        'bands': data_array,
        'band_names': [b for b in available_bands if b in band_data],
        'tile_count': 1,
        'shape': data_array.shape,
        'crs': tile_crs
    }


def process_single_tile_scenes_working(date_groups, aoi_geom, target_epsg=32643, 
                                       aoi_crs='epsg:4326', max_workers=4):
    """
    Process single-tile scenes using parallel loading and caching.
    """
    results = {}

    if not date_groups:
        print("⚠️ No date groups to process")
        return results

    # Ensure AOI is in WGS84
    if aoi_crs != 'epsg:4326':
        try:
            project_to_wgs84 = pyproj.Transformer.from_crs(
                aoi_crs, 'epsg:4326', always_xy=True
            ).transform
            aoi_wgs84 = transform(project_to_wgs84, aoi_geom)
        except:
            aoi_wgs84 = aoi_geom
    else:
        aoi_wgs84 = aoi_geom

    if aoi_wgs84.is_empty or not aoi_wgs84.is_valid:
        print("⚠️ Invalid AOI geometry")
        return results

    print(f"AOI bounds (WGS84): {aoi_wgs84.bounds}")

    # Filter to single-tile dates only
    single_dates = [(date, items) for date, items in date_groups.items() if len(items) == 1]
    if not single_dates:
        print("⚠️ No single-tile dates to process")
        return results

    print(f"\n📅 Processing {len(single_dates)} single-tile dates in parallel (max_workers={max_workers})")

    # Process dates in parallel
    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        future_to_date = {
            executor.submit(process_single_date, date, items, aoi_geom, target_epsg): date
            for date, items in single_dates
        }
        for future in as_completed(future_to_date):
            date = future_to_date[future]
            try:
                result = future.result()
                if result is not None:
                    results[date] = {
                        'bands': result['bands'],
                        'band_names': result['band_names'],
                        'tile_count': result['tile_count'],
                        'shape': result['shape'],
                        'crs': result['crs']
                    }
                    print(f"  ✓ Processed {date}: shape {result['shape']}")
            except Exception as e:
                print(f"  ✗ Failed to process {date}: {e}")

    return results


# --- Remaining functions (analyze_single_tile_dates_fixed) unchanged ---
def analyze_single_tile_dates_fixed(date_groups):
    """Analyze single-tile dates with correct band names"""
    if not date_groups:
        print("⚠️ No date groups to process")
        return [], []

    single_dates = []
    multi_dates = []

    for date, items in date_groups.items():
        if len(items) == 1:
            single_dates.append(date)
        else:
            multi_dates.append((date, len(items)))

    print(f"\n📊 Date Analysis:")
    print(f"  Total dates: {len(date_groups)}")
    print(f"  Dates with single tile: {len(single_dates)}")
    print(f"  Dates with multiple tiles: {len(multi_dates)}")

    if single_dates:
        print(f"\n📅 Single-tile dates:")
        for date in sorted(single_dates):
            item = date_groups[date][0]
            tile_name = item.id.split('_')[1] if '_' in item.id else 'unknown'
            cloud = item.properties.get('eo:cloud_cover', 'N/A')
            tile_crs = item.properties.get('proj:epsg', 'unknown')
            available = [a for a in ['blue', 'green', 'red'] if a in item.assets]
            print(f"  {date}: {tile_name} (cloud: {cloud}%, CRS: EPSG:{tile_crs}, bands: {available})")

    if multi_dates:
        print(f"\n📅 Multi-tale dates (showing first 10):")
        for date, count in sorted(multi_dates)[:10]:
            print(f"  {date}: {count} tiles")
        if len(multi_dates) > 10:
            print(f"  ... and {len(multi_dates) - 10} more")

    return single_dates, multi_dates


# ==================== MAIN EXECUTION ====================
target_epsg = 32643
single_dates, multi_dates = analyze_single_tile_dates_fixed(date_groups)

# Process with parallel loading (adjust max_workers as needed)
single_tile_results = process_single_tile_scenes_working(
    date_groups, aoi_geom, target_epsg, aoi_crs='epsg:4326', max_workers=4
)

print(f"\n✅ Processed {len(single_tile_results)} single-tile dates")

if single_tile_results:
    print("\n📊 Results:")
    for date, info in sorted(single_tile_results.items()):
        print(f"  {date}: shape {info['shape']}, bands: {info['band_names']}, CRS: EPSG:{info['crs']}")
        data = info['bands']
        print(f"    Data range: {data.min():.2f} to {data.max():.2f}")
        print(f"    Data mean: {data.mean():.2f}")
        print(f"    Data std: {data.std():.2f}")
else:
    print("  No single-tile dates were processed successfully")


📊 Date Analysis:
  Total dates: 39
  Dates with single tile: 9
  Dates with multiple tiles: 30

📅 Single-tile dates:
  2025-05-13: 43SGS (cloud: 17.851138%, CRS: EPSG:unknown, bands: ['blue', 'green', 'red'])
  2025-05-20: 44SLB (cloud: 15.451418%, CRS: EPSG:unknown, bands: ['blue', 'green', 'red'])
  2025-05-26: 43SGV (cloud: 16.954215%, CRS: EPSG:unknown, bands: ['blue', 'green', 'red'])
  2025-06-29: 44SLD (cloud: 17.494304%, CRS: EPSG:unknown, bands: ['blue', 'green', 'red'])
  2025-07-14: 44SNF (cloud: 18.581067%, CRS: EPSG:unknown, bands: ['blue', 'green', 'red'])
  2025-08-03: 44SNB (cloud: 17.132802%, CRS: EPSG:unknown, bands: ['blue', 'green', 'red'])
  2025-08-09: 43SGV (cloud: 15.488492%, CRS: EPSG:unknown, bands: ['blue', 'green', 'red'])
  2025-08-15: 44SNB (cloud: 10.836514%, CRS: EPSG:unknown, bands: ['blue', 'green', 'red'])
  2025-08-26: 43SGA (cloud: 17.363802%, CRS: EPSG:unknown, bands: ['blue', 'green', 'red'])

📅 Multi-tile dates (showing first 10):
  2025-05-01: 

KeyboardInterrupt: 

In [ ]:
import stackstac
import geopandas as gpd
import numpy as np
from shapely.geometry import shape, box, MultiPolygon
import datetime
import rioxarray
import xarray as xr
from shapely.ops import transform
import pyproj
import warnings
warnings.filterwarnings('ignore')

def process_single_date(date_groups, target_date, aoi_geom, target_epsg=32643, aoi_crs='epsg:4326'):
    """
    Process a single specific date from the date_groups
    """
    results = {}

    if not date_groups:
        print("⚠️ No date groups to process")
        return results

    # Check if the date exists
    if target_date not in date_groups:
        print(f"⚠️ Date {target_date} not found in date_groups")
        print(f"Available dates: {sorted(date_groups.keys())[:10]}...")
        return results

    # Ensure AOI is in WGS84
    if aoi_crs != 'epsg:4326':
        try:
            project_to_wgs84 = pyproj.Transformer.from_crs(
                aoi_crs, 'epsg:4326', always_xy=True
            ).transform
            aoi_wgs84 = transform(project_to_wgs84, aoi_geom)
        except:
            aoi_wgs84 = aoi_geom
    else:
        aoi_wgs84 = aoi_geom

    # Validate AOI
    if aoi_wgs84.is_empty or not aoi_wgs84.is_valid:
        print("⚠️ Invalid AOI geometry")
        return results

    print(f"AOI bounds (WGS84): {aoi_wgs84.bounds}")
    print(f"\n📅 Processing single date: {target_date}")

    items = date_groups[target_date]
    print(f"  Found {len(items)} tiles for this date")

    # Process each tile for this date (in case there are multiple)
    for idx, item in enumerate(items):
        print(f"\n  Tile {idx + 1}/{len(items)}:")

        # Get tile info
        tile_crs = item.properties.get('proj:epsg', target_epsg)
        tile_id = item.id
        tile_name = tile_id.split('_')[1] if '_' in tile_id else 'unknown'
        print(f"    Tile ID: {tile_id}")
        print(f"    Tile Name: {tile_name}")
        print(f"    Tile CRS: EPSG:{tile_crs}")

        # Use the correct band names
        band_assets = ['blue', 'green', 'red']
        available_bands = [b for b in band_assets if b in item.assets]

        if not available_bands:
            print(f"    ⚠️ No band assets found")
            continue

        print(f"    Using bands: {available_bands}")

        # Try loading each band with rioxarray
        band_data_list = []
        band_names_list = []

        for band in available_bands:
            try:
                url = item.assets[band].href
                print(f"    Loading {band} from {url}")

                # Open with rioxarray
                ds = xr.open_dataset(url, engine='rasterio')

                # Get the data - handle different data structures
                if hasattr(ds, 'band_data'):
                    data = ds.band_data.values
                elif hasattr(ds, 'data'):
                    data = ds.data.values
                else:
                    # Try to get the first data variable
                    data_vars = list(ds.data_vars.keys())
                    if data_vars:
                        data = ds[data_vars[0]].values
                    else:
                        # Direct access
                        data = ds.values

                # If data is 3D (band, height, width), take first band
                if len(data.shape) == 3:
                    data = data[0] if data.shape[0] == 1 else data

                # Check if data is valid
                if data is not None and data.size > 0:
                    band_data_list.append(data)
                    band_names_list.append(band)
                    print(f"      ✓ Loaded {band}, shape: {data.shape}")
                else:
                    print(f"      ⚠️ No data in {band}")

                ds.close()

            except Exception as e:
                print(f"      ⚠️ Failed to load {band}: {e}")
                # Try alternative method with rasterio
                try:
                    import rasterio

                    url = item.assets[band].href
                    with rasterio.open(url) as src:
                        data = src.read(1)  # Read first band
                        if data is not None and data.size > 0:
                            band_data_list.append(data)
                            band_names_list.append(band)
                            print(f"      ✓ Loaded {band} with rasterio, shape: {data.shape}")
                except Exception as e2:
                    print(f"      ⚠️ Rasterio also failed: {e2}")

        if band_data_list:
            # Stack bands
            if len(band_data_list) == 1:
                data_array = band_data_list[0]
            else:
                # Ensure all arrays have the same shape
                shapes = [d.shape for d in band_data_list]
                if len(set(shapes)) == 1:
                    data_array = np.stack(band_data_list, axis=0)
                else:
                    print(f"    ⚠️ Bands have different shapes: {shapes}")
                    # Resize to smallest shape
                    min_shape = tuple(min(dim) for dim in zip(*shapes))
                    resized_bands = []
                    for d in band_data_list:
                        if d.shape != min_shape:
                            # Crop to min shape
                            slices = tuple(slice(0, s) for s in min_shape)
                            resized_bands.append(d[slices])
                        else:
                            resized_bands.append(d)
                    data_array = np.stack(resized_bands, axis=0)

            # Store result with tile info
            tile_key = f"{target_date}_{tile_name}"
            results[tile_key] = {
                'bands': data_array,
                'band_names': band_names_list,
                'tile_count': len(items),
                'shape': data_array.shape,
                'crs': tile_crs,
                'tile_id': tile_id,
                'tile_name': tile_name,
                'date': target_date
            }
            print(f"    ✓ Success! Shape: {data_array.shape}, Bands: {band_names_list}")
        else:
            print(f"    ⚠️ Failed to load any bands for this tile")

    return results


def analyze_single_tile_dates_fixed(date_groups):
    """Analyze single-tile dates with correct band names"""

    if not date_groups:
        print("⚠️ No date groups to process")
        return [], []

    single_dates = []
    multi_dates = []

    for date, items in date_groups.items():
        if len(items) == 1:
            single_dates.append(date)
        else:
            multi_dates.append((date, len(items)))

    print(f"\n📊 Date Analysis:")
    print(f"  Total dates: {len(date_groups)}")
    print(f"  Dates with single tile: {len(single_dates)}")
    print(f"  Dates with multiple tiles: {len(multi_dates)}")

    if single_dates:
        print(f"\n📅 Single-tile dates:")
        for date in sorted(single_dates):
            item = date_groups[date][0]
            tile_name = item.id.split('_')[1] if '_' in item.id else 'unknown'
            cloud = item.properties.get('eo:cloud_cover', 'N/A')
            tile_crs = item.properties.get('proj:epsg', 'unknown')
            available = [a for a in ['blue', 'green', 'red'] if a in item.assets]
            print(f"  {date}: {tile_name} (cloud: {cloud}%, CRS: EPSG:{tile_crs}, bands: {available})")

    if multi_dates:
        print(f"\n📅 Multi-tile dates (showing first 10):")
        for date, count in sorted(multi_dates)[:10]:
            print(f"  {date}: {count} tiles")
        if len(multi_dates) > 10:
            print(f"  ... and {len(multi_dates) - 10} more")

    return single_dates, multi_dates


# Set the correct target EPSG for your area
target_epsg = 32643  # Change this to your area's UTM zone

# First, analyze the date groups
single_dates, multi_dates = analyze_single_tile_dates_fixed(date_groups)

# Choose a specific date to process (pick one from the single-tile dates)
# For example, use the first single-tile date
if single_dates:
    test_date = single_dates[0]  # Change this to any date you want to test
    print(f"\n🎯 Testing with date: {test_date}")

    # Process only this specific date
    single_tile_results = process_single_date(
        date_groups, test_date, aoi_geom, target_epsg, aoi_crs='epsg:4326'
    )

    print(f"\n✅ Processed {len(single_tile_results)} tiles for date {test_date}")

    # Display results
    if single_tile_results:
        print("\n📊 Results:")
        for tile_key, info in sorted(single_tile_results.items()):
            print(f"\n  Tile: {tile_key}")
            print(f"    Shape: {info['shape']}")
            print(f"    Bands: {info['band_names']}")
            print(f"    CRS: EPSG:{info['crs']}")
            print(f"    Tile ID: {info['tile_id']}")

            # Show some stats about the data
            data = info['bands']
            print(f"    Data range: {data.min():.2f} to {data.max():.2f}")
            print(f"    Data mean: {data.mean():.2f}")
            print(f"    Data std: {data.std():.2f}")
    else:
        print("  No tiles were processed successfully")
else:
    print("⚠️ No single-tile dates available to test")
    print("Available dates:", sorted(date_groups.keys())[:10])

In [ ]:
def process_multi_tile_scenes_safe(date_groups, aoi_geom, target_epsg):
    """Process multi-tile scenes with safe merging"""
    results = {}

    for date, items in date_groups.items():
        if len(items) > 1:
            print(f"\n{'='*60}")
            print(f"📅 Processing multi-tile date: {date} ({len(items)} tiles)")
            for item in items:
                print(f"  - {item.id}")

            bands, band_names = merge_tiles_for_date_safe(items, aoi_geom, target_epsg)

            if bands is not None:
                results[date] = {
                    'bands': bands,
                    'band_names': band_names,
                    'tile_count': len(items)
                }
                print(f"  ✅ Merged {len(items)} tiles into shape: {bands.shape}")
            else:
                print(f"  ❌ Failed to merge tiles for {date}")

    return results

# Process multi-tile scenes
multi_tile_results = process_multi_tile_scenes_safe(date_groups, aoi_geom, target_epsg)
print(f"\n✅ Processed {len(multi_tile_results)} multi-tile dates")

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
import numpy as np
import xarray as xr
import rasterio
from pathlib import Path
import geopandas as gpd
from shapely.geometry import box
import datetime
import pyproj
from shapely.ops import transform
import os

# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

def process_single_date_export_drive(date_groups, target_date, aoi_geom, target_epsg,
                                    export=True, drive_path=None):
    """
    Process and export a single specific date to Google Drive
    """
    # Convert AOI to GeoDataFrame once
    aoi_gdf = gpd.GeoDataFrame(geometry=[aoi_geom], crs='epsg:4326')

    results = {}

    # Set up output directory
    if drive_path is None:
        # Default: Save to Drive in a folder called 'Sentinel2_Outputs'
        drive_base = "/content/drive/MyDrive"
        output_dir = Path(drive_base) / "Sentinel2_Outputs" / "stitched_output_single"
    else:
        output_dir = Path(drive_path)

    output_dir.mkdir(parents=True, exist_ok=True)

    print(f"{'='*70}")
    print(f"PROCESSING SINGLE DATE: {target_date}")
    print(f"{'='*70}")
    print(f"📁 Output directory: {output_dir}")

    # Check if the date exists
    if target_date not in date_groups:
        print(f"⚠️ Date {target_date} not found in date_groups")
        print(f"Available dates: {sorted(date_groups.keys())[:10]}...")
        return results

    items = date_groups[target_date]
    print(f"📅 Found {len(items)} tiles for {target_date}")

    # Process with stitching
    bands, band_names = merge_tiles_for_date_safe(items, aoi_geom, target_epsg)

    if bands is None:
        print(f"  ✗ Failed to process {target_date}")
        return results

    results[target_date] = {
        'bands': bands,
        'band_names': band_names,
        'tile_count': len(items)
    }

    if export:
        # Export to TIFF
        filename = output_dir / f"sentinel2_{target_date.isoformat()}.tif"

        # Create alpha band
        height, width = bands.shape[1], bands.shape[2]
        alpha_band = np.ones((height, width), dtype=np.uint8)
        sum_across = np.sum(bands, axis=0)
        alpha_band[sum_across == 0] = 0

        bands_with_alpha = np.vstack([bands, alpha_band[np.newaxis, :, :]])

        # Calculate transform
        minx, miny, maxx, maxy = aoi_geom.bounds
        transform = [10, 0, minx, 0, -10, maxy]

        # Write output
        profile = {
            'driver': 'GTiff',
            'height': height,
            'width': width,
            'count': bands_with_alpha.shape[0],
            'dtype': 'float32',
            'crs': f'EPSG:{target_epsg}',
            'transform': transform,
            'compress': 'lzw',
            'tiled': True,
            'blockxsize': 512,
            'blockysize': 512,
            'predictor': 2,
            'interleave': 'pixel',
            'bigtiff': 'IF_SAFER',
            'nodata': 0
        }

        with rasterio.open(filename, 'w', **profile) as dst:
            for i in range(bands_with_alpha.shape[0]):
                dst.write(bands_with_alpha[i].astype('float32'), i+1)
            for i, name in enumerate(band_names + ['alpha'], 1):
                dst.set_band_description(i, name)

        size_mb = bands.nbytes / (1024 * 1024)
        print(f"  ✓ Exported: {filename.name} ({size_mb:.1f} MB)")

        # Print file info
        print(f"\n📁 File saved to: {filename.absolute()}")
        print(f"  Shape: {bands.shape}")
        print(f"  Bands: {band_names}")
        print(f"  CRS: EPSG:{target_epsg}")

        # Create a text file with metadata
        metadata_file = output_dir / f"sentinel2_{target_date.isoformat()}_metadata.txt"
        with open(metadata_file, 'w') as f:
            f.write(f"Sentinel-2 Output Metadata\n")
            f.write(f"{'='*50}\n")
            f.write(f"Date: {target_date}\n")
            f.write(f"Tiles used: {len(items)}\n")
            f.write(f"Band names: {band_names}\n")
            f.write(f"Shape: {bands.shape}\n")
            f.write(f"CRS: EPSG:{target_epsg}\n")
            f.write(f"Data range: {bands.min():.2f} to {bands.max():.2f}\n")
            f.write(f"Data mean: {bands.mean():.2f}\n")
            f.write(f"Data std: {bands.std():.2f}\n")
            f.write(f"File size: {size_mb:.1f} MB\n")
            f.write(f"Output file: {filename.name}\n")

            # List tile IDs
            f.write(f"\nTile IDs:\n")
            for item in items:
                f.write(f"  - {item.id}\n")

        print(f"  ✓ Metadata saved: {metadata_file.name}")

    print(f"\n✅ Successfully processed {target_date}")
    return results


def merge_tiles_for_date_safe(items, aoi_geom, target_epsg):
    """
    Merge multiple tiles for a single date, handling reprojection safely
    """
    try:
        # Reproject AOI to target EPSG
        project_aoi = pyproj.Transformer.from_crs(
            'epsg:4326', f'epsg:{target_epsg}', always_xy=True
        ).transform
        aoi_projected = transform(project_aoi, aoi_geom)

        # Get AOI bounds in target CRS
        minx, miny, maxx, maxy = aoi_projected.bounds

        # Load and clip each tile
        clipped_arrays = []

        for item in items:
            try:
                # Get the item's CRS
                item_crs = item.properties.get('proj:epsg', target_epsg)
                if isinstance(item_crs, int):
                    item_crs = f'epsg:{item_crs}'

                # Try to load band assets
                band_assets = ['blue', 'green', 'red']
                available_bands = [b for b in band_assets if b in item.assets]

                if not available_bands:
                    print(f"  ⚠️ No band assets found for {item.id}")
                    continue

                band_data = []
                for band in available_bands:
                    try:
                        url = item.assets[band].href
                        ds = xr.open_dataset(url, engine='rasterio')

                        # Get the data
                        if hasattr(ds, 'band_data'):
                            data = ds.band_data.values
                        elif hasattr(ds, 'data'):
                            data = ds.data.values
                        else:
                            data_vars = list(ds.data_vars.keys())
                            if data_vars:
                                data = ds[data_vars[0]].values
                            else:
                                data = ds.values

                        # If data is 3D, take first band
                        if len(data.shape) == 3:
                            data = data[0] if data.shape[0] == 1 else data

                        band_data.append(data)
                        ds.close()

                    except Exception as e:
                        print(f"    ⚠️ Error loading {band}: {e}")
                        continue

                if band_data:
                    # Stack bands
                    if len(band_data) == 1:
                        tile_data = band_data[0]
                    else:
                        tile_data = np.stack(band_data, axis=0)
                    clipped_arrays.append(tile_data)

            except Exception as e:
                print(f"  ⚠️ Error processing {item.id}: {e}")
                continue

        if not clipped_arrays:
            return None, None

        # Stack arrays
        if len(clipped_arrays) == 1:
            merged = clipped_arrays[0]
        else:
            # Ensure all arrays have the same shape
            shapes = [arr.shape for arr in clipped_arrays]
            if len(set(shapes)) == 1:
                merged = np.stack(clipped_arrays, axis=0)
            else:
                # Resize to smallest shape
                min_shape = tuple(min(dim) for dim in zip(*shapes))
                resized_arrays = []
                for arr in clipped_arrays:
                    if arr.shape != min_shape:
                        slices = tuple(slice(0, s) for s in min_shape)
                        resized_arrays.append(arr[slices])
                    else:
                        resized_arrays.append(arr)
                merged = np.stack(resized_arrays, axis=0)

        return merged, ['blue', 'green', 'red']

    except Exception as e:
        print(f"  ⚠️ Error merging tiles: {e}")
        return None, None


def analyze_single_tile_dates_fixed(date_groups):
    """Analyze single-tile dates with correct band names"""

    if not date_groups:
        print("⚠️ No date groups to process")
        return [], []

    single_dates = []
    multi_dates = []

    for date, items in date_groups.items():
        if len(items) == 1:
            single_dates.append(date)
        else:
            multi_dates.append((date, len(items)))

    print(f"\n📊 Date Analysis:")
    print(f"  Total dates: {len(date_groups)}")
    print(f"  Dates with single tile: {len(single_dates)}")
    print(f"  Dates with multiple tiles: {len(multi_dates)}")

    if single_dates:
        print(f"\n📅 Single-tile dates:")
        for date in sorted(single_dates):
            item = date_groups[date][0]
            tile_name = item.id.split('_')[1] if '_' in item.id else 'unknown'
            cloud = item.properties.get('eo:cloud_cover', 'N/A')
            tile_crs = item.properties.get('proj:epsg', 'unknown')
            available = [a for a in ['blue', 'green', 'red'] if a in item.assets]
            print(f"  {date}: {tile_name} (cloud: {cloud}%, CRS: EPSG:{tile_crs}, bands: {available})")

    if multi_dates:
        print(f"\n📅 Multi-tile dates (showing first 10):")
        for date, count in sorted(multi_dates)[:10]:
            print(f"  {date}: {count} tiles")
        if len(multi_dates) > 10:
            print(f"  ... and {len(multi_dates) - 10} more")

    return single_dates, multi_dates


# Set the correct target EPSG for your area
target_epsg = 32643  # Change this to your area's UTM zone

# First, analyze the date groups
single_dates, multi_dates = analyze_single_tile_dates_fixed(date_groups)

# Choose a specific date to process
if single_dates:
    target_date = single_dates[0]  # Change this to any specific date
    print(f"\n🎯 Processing date: {target_date}")

    # Process and export the single date to Drive
    # Option 1: Use default Drive location
    single_results = process_single_date_export_drive(
        date_groups,
        target_date,
        aoi_geom,
        target_epsg,
        export=True
    )

    # Option 2: Specify a custom Drive path
    # custom_path = "/content/drive/MyDrive/My_Sentinel2_Data"
    # single_results = process_single_date_export_drive(
    #     date_groups,
    #     target_date,
    #     aoi_geom,
    #     target_epsg,
    #     export=True,
    #     drive_path=custom_path
    # )

    # Display results
    if single_results:
        print(f"\n📊 Results for {target_date}:")
        for date, info in single_results.items():
            print(f"  Date: {date}")
            print(f"    Shape: {info['bands'].shape}")
            print(f"    Bands: {info['band_names']}")
            print(f"    Tiles used: {info['tile_count']}")

            # Show stats
            data = info['bands']
            print(f"    Data range: {data.min():.2f} to {data.max():.2f}")
            print(f"    Data mean: {data.mean():.2f}")
            print(f"    Data std: {data.std():.2f}")
    else:
        print("  No data was processed successfully")

else:
    print("⚠️ No single-tile dates available")

# Option: Manually specify a date
# Uncomment and modify this to process a specific date
# target_date = datetime.date(2025, 8, 26)
# single_results = process_single_date_export_drive(
#     date_groups,
#     target_date,
#     aoi_geom,
#     target_epsg,
#     export=True
# )

print("\n✅ Done! Check your Google Drive for the output files.")

## FInal SHIT

In [12]:
!uv pip install scikit-image

Audited 1 package in 8ms


In [13]:
import pyproj
os.environ['PROJ_LIB'] = pyproj.datadir.get_data_dir()

In [16]:
import os
import logging
import warnings
warnings.filterwarnings('ignore')

# --- Fix PROJ before any geospatial import ---
import pyproj
PROJ_DIR = pyproj.datadir.get_data_dir()
os.environ['PROJ_LIB'] = PROJ_DIR
os.environ['GDAL_DATA'] = PROJ_DIR
os.environ['PROJ_NETWORK'] = 'OFF'
os.environ['PROJ_DEBUG'] = '0'

# Now import rest
import datetime
import numpy as np
import xarray as xr
import rasterio
from rasterio.env import Env
from concurrent.futures import ThreadPoolExecutor, as_completed
from pathlib import Path
from collections import OrderedDict
from shapely.geometry import box
from shapely.ops import transform
import geopandas as gpd
import stackstac
import rioxarray

try:
    from skimage.transform import resize
except ImportError:
    raise ImportError("Install scikit-image: pip install scikit-image")

# --- Setup logging (no emojis, UTF-8 for file) ---
log_dir = Path.cwd() / "logs"
log_dir.mkdir(exist_ok=True)
log_file = log_dir / f"sentinel2_processing_{datetime.datetime.now().strftime('%Y%m%d_%H%M%S')}.log"

class NoEmojiFormatter(logging.Formatter):
    def format(self, record):
        msg = super().format(record)
        import re
        emoji_pattern = re.compile("["
            u"\U0001F600-\U0001F64F"
            u"\U0001F300-\U0001F5FF"
            u"\U0001F680-\U0001F6FF"
            u"\U0001F700-\U0001F77F"
            u"\U0001F780-\U0001F7FF"
            u"\U0001F800-\U0001F8FF"
            u"\U0001F900-\U0001F9FF"
            u"\U0001FA00-\U0001FA6F"
            u"\U0001FA70-\U0001FAFF"
            u"\U00002702-\U000027B0"
            u"\U000024C2-\U0001F251"
            "]+", flags=re.UNICODE)
        return emoji_pattern.sub('', msg)

logger = logging.getLogger(__name__)
logger.setLevel(logging.INFO)

fh = logging.FileHandler(log_file, encoding='utf-8')
fh.setFormatter(NoEmojiFormatter('%(asctime)s - %(name)s - %(levelname)s - %(message)s'))
logger.addHandler(fh)

ch = logging.StreamHandler()
ch.setFormatter(NoEmojiFormatter('%(asctime)s - %(name)s - %(levelname)s - %(message)s'))
logger.addHandler(ch)

# --- AWS and cache ---
os.environ['AWS_NO_SIGN_REQUEST'] = 'YES'
CACHE_DIR = Path.cwd() / "band_cache"
CACHE_DIR.mkdir(exist_ok=True)
IN_MEMORY_CACHE_MAX = 16
_mem_cache = OrderedDict()

def get_cache_path(item_id, band_name):
    safe = f"{item_id}_{band_name}.npy".replace('/', '_').replace('\\', '_').replace(':', '_')
    return CACHE_DIR / safe

def load_band_from_cache(item_id, band_name):
    path = get_cache_path(item_id, band_name)
    if path.exists():
        try:
            return np.load(path)
        except Exception:
            return None
    return None

def save_band_to_cache(item_id, band_name, data):
    path = get_cache_path(item_id, band_name)
    try:
        np.save(path, data)
    except Exception as e:
        logger.warning(f"Could not save cache for {item_id}/{band_name}: {e}")

def get_band_data(item, band_name):
    key = (item.id, band_name)
    if key in _mem_cache:
        _mem_cache.move_to_end(key)
        return _mem_cache[key]

    cached = load_band_from_cache(item.id, band_name)
    if cached is not None:
        _mem_cache[key] = cached
        if len(_mem_cache) > IN_MEMORY_CACHE_MAX:
            _mem_cache.popitem(last=False)
        return cached

    try:
        url = item.assets[band_name].href
        logger.debug(f"Downloading {band_name}")
        ds = xr.open_dataset(url, engine='rasterio')
        if hasattr(ds, 'band_data'):
            data = ds.band_data.values
        elif hasattr(ds, 'data'):
            data = ds.data.values
        else:
            data_vars = list(ds.data_vars.keys())
            if data_vars:
                data = ds[data_vars[0]].values
            else:
                data = ds.values
        if len(data.shape) == 3:
            if data.shape[0] <= 3:
                data = data[0]
            else:
                if data.shape[-1] <= 3:
                    data = data[:, :, 0]
                else:
                    data = data[0]
        ds.close()
        save_band_to_cache(item.id, band_name, data)
        _mem_cache[key] = data
        if len(_mem_cache) > IN_MEMORY_CACHE_MAX:
            _mem_cache.popitem(last=False)
        logger.info(f"Loaded {band_name} shape {data.shape}")
        return data
    except Exception as e:
        logger.warning(f"xarray failed for {band_name}: {e}, falling back to rasterio")
        try:
            with rasterio.open(url) as src:
                data = src.read(1)
            if data is not None and data.size > 0:
                save_band_to_cache(item.id, band_name, data)
                _mem_cache[key] = data
                if len(_mem_cache) > IN_MEMORY_CACHE_MAX:
                    _mem_cache.popitem(last=False)
                logger.info(f"Loaded {band_name} shape {data.shape} (rasterio)")
                return data
        except Exception as e2:
            logger.error(f"Failed to load {band_name}: {e2}")
            return None


def process_single_date_parallel(date_groups, target_date, aoi_geom, target_epsg,
                                 area_name='Leh', satellite='Sentinel2',
                                 export=True, local_path=None,
                                 max_band_workers=8,
                                 band_selection='all',
                                 crop_to_aoi=True):
    results = {}

    if local_path is None:
        output_dir = Path.cwd() / f"{area_name}_outputs"
    else:
        output_dir = Path(local_path)
    output_dir.mkdir(parents=True, exist_ok=True)

    logger.info("="*70)
    logger.info(f"PROCESSING SINGLE DATE: {target_date}")
    logger.info("="*70)
    logger.info(f"Output directory: {output_dir}")

    if target_date not in date_groups:
        logger.error(f"Date {target_date} not found")
        return results

    items = date_groups[target_date]
    if len(items) != 1:
        logger.error(f"Expected 1 tile, found {len(items)}")
        return results

    item = items[0]
    tile_crs = item.properties.get('proj:epsg', target_epsg)
    logger.info(f"Found 1 tile: {item.id} (CRS: EPSG:{tile_crs})")

    # Show all assets
    logger.info(f"Available asset keys: {list(item.assets.keys())}")

    # Select bands
    exclude = {'visual', 'thumbnail', 'tileinfo', 'metadata', 'overview', 'info', 'datastrip'}
    all_bands = [k for k in item.assets.keys() if not k.endswith('-jp2') and k not in exclude]

    if band_selection == 'all':
        available_bands = all_bands
    elif band_selection == '10m':
        desired = ['B02', 'B03', 'B04', 'B08']
        available_bands = [b for b in all_bands if b in desired]
    elif isinstance(band_selection, list):
        available_bands = [b for b in all_bands if b in band_selection]
    else:
        available_bands = all_bands

    # If no direct bands, fallback to JP2
    if not available_bands:
        logger.warning("No direct bands found, falling back to -jp2")
        available_bands = [k for k in item.assets.keys() if k.endswith('-jp2') and 'visual' not in k]

    if not available_bands:
        logger.error("No bands available")
        return results

    logger.info(f"Using bands: {available_bands}")

    # Load bands in parallel
    band_data = {}
    with ThreadPoolExecutor(max_workers=min(max_band_workers, len(available_bands))) as executor:
        future_to_band = {executor.submit(get_band_data, item, band): band for band in available_bands}
        for future in as_completed(future_to_band):
            band = future_to_band[future]
            try:
                data = future.result()
                if data is not None:
                    band_data[band] = data
            except Exception as e:
                logger.error(f"Error loading {band}: {e}")

    if not band_data:
        logger.error("No bands loaded")
        return results

    # Reference shape (use first loaded band)
    ref_band = next(iter(band_data.keys()))
    ref_shape = band_data[ref_band].shape
    logger.info(f"Reference shape: {ref_shape} (from '{ref_band}')")

    # Resample and stack
    resampled = []
    loaded_names = []
    for band in available_bands:
        if band in band_data:
            data = band_data[band]
            if data.shape != ref_shape:
                logger.info(f"Resampling {band} from {data.shape} to {ref_shape}")
                data = resize(data, ref_shape, mode='reflect', preserve_range=True).astype(data.dtype)
            resampled.append(data)
            loaded_names.append(band)

    if len(resampled) == 1:
        bands_array = resampled[0][np.newaxis, :, :]
    else:
        bands_array = np.stack(resampled, axis=0)
    logger.info(f"Stacked bands: shape {bands_array.shape}")

    # --- Get tile's geotransform and CRS from the first band ---
    first_band = available_bands[0]
    url = item.assets[first_band].href
    
    try:
        with rasterio.open(url) as src:
            src_transform = src.transform
            src_crs = src.crs
            src_bounds = src.bounds
            logger.info(f"Read geotransform and CRS from {first_band}")
    except Exception as e:
        logger.warning(f"Could not read geotransform from {first_band}: {e}")
        # Use fallback: create a default transform based on AOI
        # We'll use the tile's EPSG from properties
        src_crs = None
        src_transform = None
    
    # --- Handle missing CRS ---
    if src_crs is None:
        logger.warning("CRS not found in band, using target EPSG")
        # Use the target EPSG
        src_crs_str = f'EPSG:{target_epsg}'
    else:
        # Convert src_crs to a string that pyproj can understand
        try:
            # Try EPSG code first
            epsg_code = src_crs.to_epsg()
            if epsg_code is not None:
                src_crs_str = f'EPSG:{epsg_code}'
            else:
                # Fallback to WKT
                src_crs_str = src_crs.to_wkt()
        except Exception as e:
            logger.warning(f"Could not convert CRS: {e}, using target EPSG")
            src_crs_str = f'EPSG:{target_epsg}'
    
    logger.info(f"Using CRS: {src_crs_str}")
    
    # --- Handle missing transform ---
    if src_transform is None:
        logger.warning("Transform not found, creating from bounds")
        # Create transform from AOI bounds in target CRS
        project_aoi_to_target = pyproj.Transformer.from_crs(
            'epsg:4326', f'EPSG:{target_epsg}', always_xy=True
        ).transform
        aoi_proj = transform(project_aoi_to_target, aoi_geom)
        minx, miny, maxx, maxy = aoi_proj.bounds
        height, width = bands_array.shape[1], bands_array.shape[2]
        pixel_width = (maxx - minx) / width
        pixel_height = (maxy - miny) / height
        src_transform = rasterio.transform.from_origin(minx, maxy, pixel_width, pixel_height)
        logger.info(f"Created transform: {src_transform}")

    # --- Crop to AOI if requested ---
    if crop_to_aoi and src_transform is not None:
        try:
            # Reproject AOI to tile CRS
            project_to_tile = pyproj.Transformer.from_crs(
                'epsg:4326', src_crs_str, always_xy=True
            ).transform
            aoi_tile = transform(project_to_tile, aoi_geom)
            minx, miny, maxx, maxy = aoi_tile.bounds
            
            # Convert bounds to pixel indices
            col_min = int((minx - src_transform.c) / src_transform.a)
            col_max = int((maxx - src_transform.c) / src_transform.a)
            row_min = int((maxy - src_transform.f) / src_transform.e)
            row_max = int((miny - src_transform.f) / src_transform.e)
            
            # Ensure within image bounds
            height, width = bands_array.shape[1], bands_array.shape[2]
            col_min = max(0, min(col_min, width))
            col_max = max(0, min(col_max, width))
            row_min = max(0, min(row_min, height))
            row_max = max(0, min(row_max, height))
            
            if col_max <= col_min or row_max <= row_min:
                logger.warning("AOI does not overlap with tile. Using full tile.")
                crop_to_aoi = False
            else:
                bands_array = bands_array[:, row_min:row_max, col_min:col_max]
                logger.info(f"Cropped to AOI: new shape {bands_array.shape}")
                
                # Update transform
                new_transform = rasterio.transform.from_origin(
                    src_transform.c + col_min * src_transform.a,
                    src_transform.f + row_min * src_transform.e,
                    src_transform.a,
                    abs(src_transform.e)
                )
                geotransform = new_transform
                height, width = bands_array.shape[1], bands_array.shape[2]
        except Exception as e:
            logger.warning(f"Error cropping to AOI: {e}. Using full tile.")
            crop_to_aoi = False

    if not crop_to_aoi:
        # Use full tile
        geotransform = src_transform
        height, width = bands_array.shape[1], bands_array.shape[2]

    # Use pyproj for CRS WKT
    from pyproj import CRS as pyproj_CRS
    crs_pyproj = pyproj_CRS.from_epsg(target_epsg)
    crs_wkt = crs_pyproj.to_wkt()

    # --- Export ---
    if export:
        filename = output_dir / f"{area_name}_{target_date.isoformat()}_{satellite}.tif"
        logger.info(f"Writing GeoTIFF to {filename}")

        # Alpha band
        sum_bands = np.nansum(bands_array, axis=0)
        alpha = np.ones((height, width), dtype=np.uint8)
        alpha[sum_bands == 0] = 0
        bands_with_alpha = np.vstack([bands_array, alpha[np.newaxis, :, :]])

        profile = {
            'driver': 'GTiff',
            'height': height,
            'width': width,
            'count': bands_with_alpha.shape[0],
            'dtype': 'float32',
            'crs': crs_wkt,
            'transform': geotransform,
            'compress': 'lzw',
            'tiled': True,
            'blockxsize': 512,
            'blockysize': 512,
            'predictor': 2,
            'interleave': 'pixel',
            'bigtiff': 'IF_SAFER',
            'nodata': 0
        }

        with Env(PROJ_LIB=PROJ_DIR, GDAL_DATA=PROJ_DIR):
            try:
                with rasterio.open(filename, 'w', **profile) as dst:
                    for i in range(bands_with_alpha.shape[0]):
                        dst.write(bands_with_alpha[i].astype('float32'), i+1)
                    for i, name in enumerate(loaded_names + ['alpha'], 1):
                        dst.set_band_description(i, name)

                size_mb = bands_array.nbytes / (1024 * 1024)
                logger.info(f"Exported: {filename.name} ({size_mb:.1f} MB)")
                logger.info(f"Saved to: {filename.absolute()}")

                # Metadata
                metadata_file = output_dir / f"{area_name}_{target_date.isoformat()}_{satellite}_metadata.txt"
                with open(metadata_file, 'w') as f:
                    f.write(f"Sentinel-2 Output Metadata\n{'='*50}\n")
                    f.write(f"Area: {area_name}\nSatellite: {satellite}\nDate: {target_date}\n")
                    f.write(f"Tile: {item.id}\nCRS: EPSG:{target_epsg}\n")
                    f.write(f"Bands: {loaded_names}\nShape: {bands_array.shape}\n")
                    if geotransform is not None:
                        f.write(f"Pixel size: {geotransform.a:.3f} m\n")
                    finite_data = bands_array[np.isfinite(bands_array)]
                    if finite_data.size > 0:
                        f.write(f"Data range: {finite_data.min():.2f} - {finite_data.max():.2f}\n")
                        f.write(f"Mean: {finite_data.mean():.2f}\nStd: {finite_data.std():.2f}\n")
                    else:
                        f.write("No valid data (all NaN)\n")
                    f.write(f"File size: {size_mb:.1f} MB\nOutput: {filename.name}\n")
                logger.info(f"Metadata saved: {metadata_file.name}")
            except Exception as e:
                logger.error(f"Failed to write GeoTIFF: {e}")
                import traceback
                logger.error(traceback.format_exc())
                return results

    results[target_date] = {
        'bands': bands_array,
        'band_names': loaded_names,
        'shape': bands_array.shape,
        'crs': target_epsg
    }

    logger.info(f"Successfully processed {target_date}")
    return results


# ==================== MAIN ====================
target_epsg = 32643 
#  2025-05-20
target_date = datetime.date(2025, 5, 20)

results = process_single_date_parallel(
    date_groups=date_groups,
    target_date=target_date,
    aoi_geom=aoi_geom,
    target_epsg=target_epsg,
    area_name='Leh',
    satellite='Sentinel2',
    export=True,
    local_path=None,
    max_band_workers=8,
    band_selection='10m',   # or 'all'
    crop_to_aoi=True
)

if results:
    for date, info in results.items():
        logger.info(f"\nResults for {date}:")
        logger.info(f"  Shape: {info['shape']}")
        logger.info(f"  Bands: {info['band_names']}")
        logger.info(f"  CRS: EPSG:{info['crs']}")
        data = info['bands']
        finite = data[np.isfinite(data)]
        if finite.size > 0:
            logger.info(f"  Range: {finite.min():.2f} - {finite.max():.2f}")
            logger.info(f"  Mean: {finite.mean():.2f}, Std: {finite.std():.2f}")
        else:
            logger.info("  No valid data (all NaN)")
else:
    logger.error("No data processed.")

2026-06-29 15:59:18,787 - __main__ - INFO - ======================================================================
2026-06-29 15:59:18,787 - __main__ - INFO - ======================================================================
2026-06-29 15:59:18,787 - __main__ - INFO - ======================================================================
2026-06-29 15:59:18,791 - __main__ - INFO - PROCESSING SINGLE DATE: 2025-05-20
2026-06-29 15:59:18,791 - __main__ - INFO - PROCESSING SINGLE DATE: 2025-05-20
2026-06-29 15:59:18,791 - __main__ - INFO - PROCESSING SINGLE DATE: 2025-05-20
2026-06-29 15:59:18,794 - __main__ - INFO - ======================================================================
2026-06-29 15:59:18,794 - __main__ - INFO - ======================================================================
2026-06-29 15:59:18,794 - __main__ - INFO - ======================================================================
2026-06-29 15:59:18,796 - __main__ - INFO - Output directory: C:\Users\Sh